###Run Shared Libraries

In [0]:
%run ../Misc/sharedlibraries

In [0]:
UpdatedDateTime = datetime.datetime.now()
Entity = "dimcusttable"

###Read Bronze tables

In [0]:
custtablebronzedf = spark.table("bronze.custtable")

###Build Dimension/Fact table

In [0]:
custtabledf = custtablebronzedf.filter(custtablebronzedf.RecordId.isNotNull()
    ).select(
        custtablebronzedf.CustomerId,
        F.when(custtablebronzedf.LastProcessedChange_DateTime.isNull(), F.lit("1900-01-01")).otherwise(custtablebronzedf.LastProcessedChange_DateTime).cast("timestamp").alias("LastProcessedChange_DateTime"),
        F.from_utc_timestamp(custtablebronzedf.DataLakeModified_DateTime,'CST').alias("DataLakeModified_DateTime"),
        F.trim(custtablebronzedf.CustomerName).alias("CustomerName"),
        F.trim(custtablebronzedf.Email).alias("Email"),
        F.trim(custtablebronzedf.Phone).alias("Phone"),
        F.trim(custtablebronzedf.Address).alias("Address"),
        F.trim(custtablebronzedf.City).alias("City"),
        F.trim(custtablebronzedf.State).alias("State"),
        F.trim(custtablebronzedf.Country).alias("Country"),
        F.trim(custtablebronzedf.ZipCode).alias("ZipCode"),
        F.trim(custtablebronzedf.Region).alias("Region"),
        F.from_utc_timestamp(custtablebronzedf.SignupDate,'CST').alias("SignupDate"),
        custtablebronzedf.RecordId.alias("CustRecordId")
    ).withColumn("UpdatedDateTime", F.lit(UpdatedDateTime)
    ).withColumn("CustomerHashKey", F.xxhash64("CustRecordId")
    )

display(custtabledf)

###Final dataframe

In [0]:
df_final = custtabledf

## Write to Silver Schema

In [0]:
saveDeltaTableToCatalog(df_final,"silver",Entity)